# 04 — Page extraction: clean -> extract -> confidence -> report

> **Restart kernel + Run All.** After editing any `.py` file: restart the kernel,
> then Run All. This notebook deliberately does not use `%autoreload` — reloading a
> module mid-session leaves two copies of the same Pydantic class alive and breaks
> validation (`Input should be ... instance of CallUsage`).

> **Cost gate:** no cell calls Gemini unless `RUN_LIVE_LLM=1` is set in the
> environment. Unset -> the whole notebook runs on the deterministic fake provider.

**What this turns into:** HTML/feed text -> one `Opportunity` per item, with

- title, organizer, type, location, eligibility, benefits, **deadline**
  (ISO `YYYY-MM-DD`), source URL, `source_checked` timestamp and a **confidence per field**;
- fields below `CONFIDENCE_THRESHOLD` — and any missing deadline — marked **"to verify"**;
- absent/rolling deadlines (`ongoing`, `present`, `current`) stored as **null**, never
  invented; unparseable claims become null + flagged;
- a failing page, feed or LLM call recorded as a failure — the run always finishes.

Steps 1-4 run offline on `data/samples/*.html` + the sample feed. Steps 5-7 are the
live steps: with `RUN_LIVE_LLM=1` (+ the matching key) they fetch a real page and
extract real feed entries; without it they take their clearly labeled OFFLINE branch,
so the same code is exercised either way.

> **Privacy:** public opportunity pages and feed entries MAY be sent to Gemini.
> Personal documents (CVs, certificates, portfolios) stay private — samples and
> fixtures are synthetic only (see `README.md`).


In [ ]:
import json
import os
import sys
from pathlib import Path

# Make oppradar_ai importable whether the kernel starts in ai/ or ai/notebooks/.
AI_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(AI_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_ROOT))

import pandas as pd
from IPython.display import display

from oppradar_ai.config import Settings, get_settings, setup_logging
from oppradar_ai.ingestion import (
    PageFetcher,
    RunBudget,
    clean_html,
    extract_from_text,
    extract_record,
    fields_filled,
    load_sources,
    parse_deadline,
    run_ingestion,
)
from oppradar_ai.llm import get_llm
from oppradar_ai.schemas import OpportunityExtraction, Source

# ── Cost gate: no cell may call Gemini unless you opt in ──────────────────────
# RUN_LIVE_LLM=1 -> the provider configured in ai/.env is used (its answers are
# disk-cached, so a repeat run is free). Unset -> the whole notebook runs on the
# deterministic fake provider: steps 2/4 can never spend a free-tier call, and
# steps 5/6/7 take their clearly labeled OFFLINE branch.
RUN_LIVE_LLM = Settings().run_live_llm   # the switch the pipeline reads too
os.environ.pop('LLM_PROVIDER', None)     # drop the force of a previous run
_configured = Settings().llm_provider    # fresh read of ai/.env (get_settings is cached)
# A live provider must be able to answer here: Gemini needs a key.
_can_live = _configured != 'gemini' or bool(Settings().gemini_key_value)
os.environ['LLM_PROVIDER'] = _configured if RUN_LIVE_LLM and _can_live else 'fake'
get_settings.cache_clear()               # settings are cached per process

setup_logging('INFO')
settings = get_settings().ensure_dirs()
SAMPLES = settings.samples_dir

# REAL_MODE: step 5 does one polite HTTP GET (needs the Tavily key).
# REAL_LLM : steps 6/7 send PUBLIC feed text to Gemini (allowed — README privacy).
REAL_MODE = RUN_LIVE_LLM and bool(settings.tavily_key_value)
REAL_LLM = settings.llm_provider == 'gemini' and bool(settings.gemini_key_value)
print('live LLM:', 'yes' if REAL_LLM else 'no',
      '| provider:', settings.llm_provider,
      '| steps 5/7:', 'LIVE' if RUN_LIVE_LLM else 'OFFLINE')


## Step 1 — Cleaning: raw HTML -> readable text

`clean_html` keeps the page title, the canonical permalink and the article body
(nav/scripts/cookie banners dropped) and collects outbound links. Garbage in ->
empty fields out, never an exception.

In [ ]:
pages = {}
for file in ('page_scholarship.html', 'page_hackathon.html'):
    pages[file] = clean_html((SAMPLES / file).read_text(encoding='utf-8'), str(SAMPLES / file))

clean_df = pd.DataFrame([
    {
        'file': file,
        'title': page.title[:52],
        'canonical url': page.canonical,
        'text chars': len(page.text),
        'links': len(page.links),
        'deadline in text': ('15 March 2027' in page.text) or ('2027-04-10' in page.text),
    }
    for file, page in pages.items()
])
display(clean_df)

## Step 2 — One budgeted extraction per page

One extraction per page, counted against `MAX_EXTRACTIONS_PER_RUN` (default 20; disk-cache
hits are free). Rules decide everything the model may not guess. The confidence table
shows which fields are marked "to verify".

**Cost:** with `RUN_LIVE_LLM` unset this runs on the fake provider, so it can never spend
a free-tier call.


In [ ]:
budget = RunBudget(settings)
opps = []
for file, page in pages.items():
    opp = extract_from_text(
        page.text,
        url=page.canonical or str(SAMPLES / file),
        source_name='samples',
        title_hint=page.title,
        budget=budget,
        settings=settings,
    )
    if opp is not None:
        opps.append(opp)

fields_df = pd.DataFrame([
    {
        'title': o.title[:44],
        'organizer': o.organizer[:28],
        'type': o.type.value,
        'deadline (ISO)': str(o.deadline) if o.deadline else '(null - never invented)',
        'fields filled': fields_filled(o),
        'to verify': ', '.join(fc.field for fc in o.to_verify()) or '-',
        'url': o.url,
        'source_checked': o.source_checked.isoformat() if o.source_checked else '',
    }
    for o in opps
])
display(fields_df)
print('provider:', settings.llm_provider, '| LLM usage for this step:',
      budget.extractions.model_dump())

if opps:
    conf_df = pd.DataFrame([
        {
            'field': fc.field,
            'confidence': fc.confidence,
            'status': 'TO VERIFY' if fc.needs_review else 'ok',
        }
        for fc in opps[0].field_confidences
    ])
    display(conf_df)


## Step 3 — Deadline normalization (ISO, never invented)

Raw text -> `date` or null. Rolling phrasing -> null; a month without a day -> null
(no guessing); impossible dates -> null; all-numeric dates read as day-first
(`11/30/2026` still resolves because 30 > 12).

In [ ]:
deadline_demo = pd.DataFrame([
    {'raw (as written)': '15 March 2027', 'ISO / null': parse_deadline('15 March 2027')},
    {'raw (as written)': '2027-04-10', 'ISO / null': parse_deadline('2027-04-10')},
    {'raw (as written)': '30/11/2026', 'ISO / null': parse_deadline('30/11/2026')},
    {'raw (as written)': '15 mars 2027', 'ISO / null': parse_deadline('15 mars 2027')},
    {'raw (as written)': 'ongoing', 'ISO / null': parse_deadline('ongoing') or '(null)'},
    {'raw (as written)': 'present', 'ISO / null': parse_deadline('present') or '(null)'},
    {'raw (as written)': 'March 2027 (no day)', 'ISO / null': parse_deadline('March 2027') or '(null)'},
    {'raw (as written)': 'sometime next year', 'ISO / null': parse_deadline('sometime next year') or '(null)'},
])
display(deadline_demo)

## Step 4 — A whole run: per-source table, usage and failures

`run_ingestion` over the two sample pages + the sample feed + one deliberately broken
path:

| what | where it lands |
|---|---|
| items found per source | `per_source[].fetched / extracted / new` |
| fields filled per source | `per_source[].fields_filled` |
| failures | `per_source[].errors` and `report.failures` |
| remote calls | `report.searches` / `report.extractions` (`made` / `cached` / `skipped`) |

The broken source proves the never-crash rule: logged, reported, zero exceptions.

**Cost:** same gate as step 2 — `RUN_LIVE_LLM` unset means the fake provider, so this
whole run is free.


In [ ]:
sample_sources = [
    Source(name='sample-feed', url=str(SAMPLES / 'feed_opportunities.xml'), kind='rss'),
    Source(name='sample-scholarship', url=str(SAMPLES / 'page_scholarship.html'), kind='page'),
    Source(name='sample-hackathon', url=str(SAMPLES / 'page_hackathon.html'), kind='page'),
    Source(name='broken-source', url=str(SAMPLES / 'missing_page.html'), kind='page'),
]
report = run_ingestion(sources=sample_sources, settings=settings)

per_source_df = pd.DataFrame([
    {
        'source': s.source,
        'items found': s.fetched,
        'extracted': s.extracted,
        'fields filled': s.fields_filled,
        'new': s.new,
        'failures': '; '.join(s.errors) or '-',
    }
    for s in report.per_source
])
display(per_source_df)

usage_df = pd.DataFrame([
    {'call kind': 'Tavily searches', **report.searches.model_dump()},
    {'call kind': 'LLM extractions', **report.extractions.model_dump()},
])
display(usage_df)

print(
    f'totals: fetched={report.items_fetched} new={report.items_new} '
    f'duplicates={report.items_duplicate} flagged={report.items_flagged}'
)
print(
    f'accepted={report.items_new} rejected={report.items_rejected} '
    f'list_pages={report.items_list_pages} detail_fetches={report.detail_fetches}'
)
print(
    f'deadlines: from_rule={report.deadline_from_rule} from_llm={report.deadline_from_llm} '
    f'mismatch={report.deadline_mismatches} null={report.deadline_null}'
)
if report.rejected:
    for r in report.rejected:
        print(f'  rejected: {r.title[:48]!r} -> {r.reason}')
for failure in report.failures:
    print('failure:', failure)

## Step 5 — Live politeness check (needs `RUN_LIVE_LLM=1` + `TAVILY_API_KEY`)

One real fetch of the **first enabled page source** in `sources.yaml` through
`PageFetcher` (User-Agent, delay, robots.txt) — the cleaned text is shown here.
That text is public opportunity data, so it **MAY be sent to Gemini** later
(`README.md` privacy rule); personal documents stay private. Offline this cell
prints `skipped`.


In [ ]:
if REAL_MODE:
    target = next((s for s in load_sources() if s.enabled and s.kind == 'page'), None)
    if target is None:
        print('no enabled page source in sources.yaml')
    else:
        doc = PageFetcher(settings).get(target.url)
        if doc.ok:
            page = clean_html(doc.html, target.url)
            print(f'{target.name}: HTTP {doc.status}, {len(page.text)} chars cleaned')
            print('title:', page.title)
            print('(public page text — MAY be sent to Gemini; this step only '
                  'fetches and cleans it)')
        else:
            print(f'{target.name}: {doc.error} -> recorded as a failure, no crash')
else:
    print('skipped (OFFLINE mode): set RUN_LIVE_LLM=1 + TAVILY_API_KEY to run this fetch')


## Step 6 — Real extraction on live feed entries

This is the proof that Gemini extracts real opportunities end-to-end. It reads the
**frozen snapshot** (`data/eval/live_snapshot.json`, created by
`scripts/snapshot_live.py`), picks **12 entries round-robin** across the feeds, and
extracts from the **RSS title + summary only** (truncated to `FEED_TEXT_MAX_CHARS`,
default 6000). Budget cap 12, disk-cache on.

For each entry it prints **which provider + model produced the record** (read from
the LLM cache record, so you can see it is not the fake), plus the type,
`is_opportunity`, the raw deadline phrase from the source, the extracted ISO date,
per-field confidences, the fields marked "to verify", and the first 300 chars of the
source text.

**OFFLINE branch (no `RUN_LIVE_LLM=1`):** the identical code runs on the fake
provider, so a Run All without any key still exercises this cell — every row then
shows provider `fake`. Refresh the snapshot any time with
`python scripts/snapshot_live.py`.


In [ ]:
snapshot = json.loads((settings.eval_dir / 'live_snapshot.json').read_text(encoding='utf-8'))
entries = snapshot['entries']

# Round-robin across the sources in the snapshot so one feed never dominates.
by_source = {}
for e in entries:
    by_source.setdefault(e['source'], []).append(e)
picked, i = [], 0
while len(picked) < 12 and any(i < len(v) for v in by_source.values()):
    for src in sorted(by_source):
        if i < len(by_source[src]) and len(picked) < 12:
            picked.append(by_source[src][i])
    i += 1

system = (settings.prompts_dir / 'extract_opportunity.md').read_text(encoding='utf-8')
llm = get_llm()   # the configured provider; the cost gate forced 'fake' when offline
mode = 'LIVE Gemini' if REAL_LLM else 'OFFLINE fake provider (no live call)'
print(f'Step 6 mode: {mode} | provider: {llm.name} | model: {llm.model}')
budget = RunBudget(settings, max_extractions=12)
rows = []
for e in picked:
    text = (e['title'] + '\n' + e['summary'])[: settings.feed_text_max_chars]
    out = extract_record(
        text, url=e['url'], source_name=e['source'],
        title_hint=e['title'], budget=budget, settings=settings,
    )
    meta = llm.cached_meta(text, OpportunityExtraction, system=system) or {}
    ext, opp = out.extraction, out.opportunity
    rows.append({
        # e01, e02, ...: the snapshot entry id, so you can label this exact
        # row in step 7's live_labels.json.
        'id': e['id'],
        # '?' -> no cache record: the free-tier quota was spent or the
        # provider failed, so this row is a rules-only fallback.
        'provider': meta.get('provider') or 'rules-only',
        'model': meta.get('model') or '-',
        # the is_opportunity gate is an LLM judgement: without an LLM
        # answer there is no verdict (every field is 'to verify').
        'is_opp': (out.rejected_reason is None) if meta else '(needs LLM)',
        'title': (ext.title if ext else (opp.title if opp else '?'))[:44],
        'type': ext.type.value if ext else '?',
        'deadline (source phrase)': (ext.deadline or '(none)') if ext else '?',
        'deadline ISO': str(opp.deadline) if opp and opp.deadline else '(null)',
        'conf ok': f"{sum(1 for fc in opp.field_confidences if not fc.needs_review)}"
                   f"/{len(opp.field_confidences)}" if opp else '-',
        'to verify': ', '.join(fc.field for fc in opp.to_verify()) if opp else '-',
        'rejected reason': out.rejected_reason or '-',
        'source text (300 chars)': text[:300].replace('\n', ' '),
    })
display(pd.DataFrame(rows))
print('budget:', budget.extractions.model_dump())


## Step 7 — Score the extraction against YOUR hand labels

`scripts/snapshot_live.py` also writes `data/eval/live_labels.template.json`.
Copy it to `data/eval/live_labels.json` and fill in, **by hand**, for each entry:
`is_opportunity` (true/false), `deadline` (ISO `YYYY-MM-DD`, or `null` when the source
states no/rolling deadline) and `type`.

This cell then re-extracts every labelled entry and reports accuracy for
`is_opportunity`, `deadline` (exact match, with **invented deadlines** — a date the
source never gave — counted separately as the worst error) and `type`, and lists every
wrong row next to its source text. If the labels file does not exist it prints how to
create one.

**OFFLINE branch (no `RUN_LIVE_LLM=1`):** with no labels to grade against, the cell
grades the committed all-null template with the fake provider — the numbers below
then prove the scoring code runs, they are **not** a measurement.


In [ ]:
labels_path = settings.eval_dir / 'live_labels.json'
template_path = settings.eval_dir / 'live_labels.template.json'

labels = None
if REAL_LLM and labels_path.exists():
    labels = json.loads(labels_path.read_text(encoding='utf-8'))
    print(f'grading against your hand labels: {len(labels)} rows')
elif REAL_LLM:
    print('No hand labels yet. To create them:')
    print('  1. copy data/eval/live_labels.template.json -> data/eval/live_labels.json')
    print('  2. fill every row: is_opportunity true/false, deadline (ISO or null), type')
else:
    # OFFLINE branch: the committed all-null template keeps the scoring code below
    # exercised without a key. Its numbers are a smoke test, not a measurement.
    labels = json.loads(template_path.read_text(encoding='utf-8'))
    print(f'OFFLINE mode (fake provider): grading {len(labels)} rows against the '
          'all-null TEMPLATE - smoke test, not a measurement')

if labels:
    budget = RunBudget(settings, max_extractions=20)
    snapshot = json.loads((settings.eval_dir / 'live_snapshot.json').read_text(encoding='utf-8'))
    entries = {e['id']: e for e in snapshot['entries']}

    total = ok_opp_n = ok_dl_n = ok_type_n = invented = 0
    wrong = []
    for eid, gold in labels.items():
        e = entries.get(eid)
        if e is None:
            continue
        text = (e['title'] + '\n' + e['summary'])[: settings.feed_text_max_chars]
        out = extract_record(
            text, url=e['url'], source_name=e['source'],
            title_hint=e['title'], budget=budget, settings=settings,
        )
        opp = out.opportunity
        pred_opp = out.rejected_reason is None
        pred_dl = str(opp.deadline) if opp and opp.deadline else None
        pred_type = opp.type.value if opp else None

        gold_opp = bool(gold['is_opportunity'])
        gold_dl = gold['deadline']
        gold_type = gold['type']

        total += 1
        good_opp = pred_opp == gold_opp
        ok_opp_n += good_opp
        if gold_dl is None and pred_dl is not None:
            invented += 1          # worst error: a deadline the source never gave
            good_dl = False
        else:
            good_dl = pred_dl == gold_dl
        ok_dl_n += good_dl
        good_type = pred_type == gold_type
        ok_type_n += good_type

        if not (good_opp and good_dl and good_type):
            wrong.append({
                'id': eid, 'source': e['source'], 'title': e['title'][:40],
                'gold is_opp': gold_opp, 'pred is_opp': pred_opp,
                'gold deadline': gold_dl, 'pred deadline': pred_dl,
                'gold type': gold_type, 'pred type': pred_type,
                'source text (200 chars)': e['summary'][:200],
            })

    print(f'is_opportunity accuracy : {ok_opp_n}/{total}')
    print(f'deadline (exact match)  : {ok_dl_n}/{total}   invented deadlines: {invented}')
    print(f'type accuracy           : {ok_type_n}/{total}')
    if wrong:
        display(pd.DataFrame(wrong))
    else:
        print('every labelled row matches')


## Expected output

- **top of the notebook**: `live LLM: no` unless `RUN_LIVE_LLM=1` (+ key) is set;
- **cleaning table**: 2 files, canonical URLs, 800+ chars each, deadline present;
- **fields table**: 2 opportunities with ISO deadlines or `(null - never invented)`,
  `to verify` listing the low-confidence fields, provider `fake` when the gate is off,
  usage `made: 2` on a cold cache (or `cached: 2` free when an earlier run already
  extracted the samples);
- **confidence table**: 9 rows, `location` marked TO VERIFY on the fake output;
- **deadline demo**: dates parse to ISO; `ongoing` / `present` / `March 2027` /
  `sometime next year` show `(null)`;
- **report table**: feed = 3 items, 2 pages = 1 each, `broken-source` shows
  `local file not found`; totals also show `accepted / rejected / flagged`, the
  deadline provenance (`rule / llm / mismatch / null`) and `detail fetches 0`
  offline — a missing deadline logs `detail fetch skipped (offline)` instead of
  requesting a page (live: up to `MAX_DETAIL_FETCHES_PER_RUN` real fetches);
  with the fake provider the Green Shield feed item takes its deadline from the
  `Application Deadline:` **rule** (2026-11-30) and records a `deadline mismatch`
  against the fake's canned date — the rule wins;
- **step 5**: the first **enabled** page source (`campusfrance-eiffel`) fetched live
  with `RUN_LIVE_LLM=1` + `TAVILY_API_KEY`, otherwise `skipped (OFFLINE mode)`;
- **step 6**: a 12-row table whose first column is the snapshot `id` (`e01`,
  `e02`, ...) — paste it into `live_labels.json` to label that exact row — and
  every row showing the **provider + model** that
  produced it — `gemini / gemini-3.5-flash` live, `fake / fake` offline. Live, the
  raw deadline phrase from the source sits next to the ISO date and listicles
  (`Top 10 ...`) come back `is_opp: False` with a reason;
- **step 7**: live + your hand labels -> accuracy for `is_opportunity`, `deadline`
  and `type`, invented deadlines counted separately, and every wrong row next to its
  source text; live without labels -> how to create them; offline -> the same scoring
  code graded against the all-null template (smoke test);
- offline (fake / no keys): everything above runs deterministically on the committed
  fixtures — steps 6/7 take their OFFLINE branch instead of skipping;
- nothing raises — a failing page, feed or LLM call becomes a message.
